In [2]:
a=3
print(a)

3


In [4]:
import sys, os
from PyQt6.QtWidgets import QApplication, QMainWindow, QTextEdit, QMessageBox, QFileDialog
from PyQt6.QtGui import QAction, QKeySequence, QFont

class Cahier(QMainWindow):
    BASE_FONT_SIZE = 11
    def __init__(self):
        super().__init__()
        self.current_file: str | None = None
        self.font_family = "맑은 고딕"
        self.font_size = self.BASE_FONT_SIZE

        self._build_editor()
        self._build_menu()
        self.setWindowTitle("cahier")
        self.resize(800, 600)

    def _build_editor(self):
        self.editor = QTextEdit()
        self.editor.setFont(QFont(self.font_family, self.font_size))
        self.editor.setLineWrapMode(QTextEdit.LineWrapMode.WidgetWidth)
        self.setCentralWidget(self.editor)


    def _build_menu(self):
        mb = self.menuBar()
        fm = mb.addMenu("파일")
        fm.addAction(self._act("새로 만들기(&N)", "Ctrl+N", self.new_file))
        fm.addAction(self._act("열기(&O)...",     "Ctrl+O", self.open_file))
        fm.addAction(self._act("저장(&S)",        "Ctrl+S", self.save_file))
        fm.addAction(self._act("다른 이름으로 저장(&A)...", None, self.save_as))
        fm.addSeparator()
        fm.addAction(self._act("종료(&X)",        "Alt+F4", self.close))

    # ── 액션 헬퍼 ────────────────────────────

    def _act(self, label: str, shortcut: str | None, slot) -> QAction:
        action = QAction(label, self)
        if shortcut:
            action.setShortcut(QKeySequence(shortcut))
        action.triggered.connect(slot)
        return action

    # ── 파일 작업 ────────────────────────────

    def new_file(self):
        if not self._check_save():
            return
        self.editor.clear()
        self.current_file = None
        self.setWindowTitle("메모장")
        self.editor.document().setModified(False)

    def open_file(self):
        if not self._check_save():
            return
        path, _ = QFileDialog.getOpenFileName(
            self, "열기", "",
            "텍스트 파일 (*.txt);;모든 파일 (*.*)"
        )
        if path:
            with open(path, encoding="utf-8") as f:
                self.editor.setPlainText(f.read())
            self.current_file = path
            self.setWindowTitle(f"{os.path.basename(path)} - 메모장")
            self.editor.document().setModified(False)

    def save_file(self):
        if self.current_file:
            with open(self.current_file, "w", encoding="utf-8") as f:
                f.write(self.editor.toPlainText())
            self.editor.document().setModified(False)
            # 제목의 * 제거
            t = self.windowTitle()
            if t.startswith("*"):
                self.setWindowTitle(t[1:])
        else:
            self.save_as()
    
    def save_as(self):
        path, _ = QFileDialog.getSaveFileName(
            self, "다른 이름으로 저장", "",
            "텍스트 파일 (*.txt);;모든 파일 (*.*)"
        )
        if path:
            self.current_file = path
            self.save_file()
            self.setWindowTitle(f"{os.path.basename(path)} - 메모장")

    # ── 내부 유틸 ─────────────────────────────


    def _check_save(self) -> bool:
        """저장 확인. 계속 진행해도 되면 True 반환."""
        if not self.editor.document().isModified():
            return True
        ans = QMessageBox.question(
            self, "저장",
            "저장하지 않은 내용이 있습니다. 저장할까요?",
            QMessageBox.StandardButton.Yes |
            QMessageBox.StandardButton.No  |
            QMessageBox.StandardButton.Cancel,
        )
        if ans == QMessageBox.StandardButton.Yes:
            self.save_file()
            return True
        if ans == QMessageBox.StandardButton.No:
            return True
        return False  # Cancel

    def closeEvent(self, event):
        if self._check_save():
            event.accept()
        else:
            event.ignore()

app = QApplication(sys.argv)
win = Cahier()
win.show()
app.exec()

: 